# 03 · CNN From Scratch  ·  *Role C*

**Goal (Part 2, model 1):** a compact **siamese / two-branch** CNN built from scratch that
takes the (pre, post) patch pair and predicts the 4-level damage class.
Keep it small — a deep net will overfit the rare classes.

In [ ]:
import sys, os; sys.path.append(os.path.abspath('..'))
from src import config, data_utils, models, train_utils
import torch
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

### Data loaders (train / val on seen disasters)

In [ ]:
train_ds = data_utils.DamageDataset(split='train', augment=True)
val_ds   = data_utils.DamageDataset(split='val',   augment=False)
train_dl = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=config.BATCH_SIZE, shuffle=False, num_workers=0)
print(len(train_ds), 'train /', len(val_ds), 'val patches')

### Model — siamese two-branch CNN (shared weights)
Defined in `src/models.py` as `SiameseScratchCNN`. Two image branches share weights;
their feature vectors are combined (concat + abs-difference) and sent to a small MLP head.

In [ ]:
model = models.SiameseScratchCNN(num_classes=config.NUM_CLASSES).to(device)
print(model)
print('params:', sum(p.numel() for p in model.parameters())/1e6, 'M')

### Train
Use the shared `train_utils.train_model` with class-weighted cross-entropy
(weights from Role A's `class_weights.npy`).

In [ ]:
import numpy as np
from pathlib import Path
w_path = Path(config.DATA_DIR)/'processed'/'class_weights.npy'
class_weights = torch.tensor(np.load(w_path), dtype=torch.float32) if w_path.exists() else None

history = train_utils.train_model(model, train_dl, val_dl, device,
                                  epochs=config.EPOCHS, class_weights=class_weights,
                                  ckpt_path='../checkpoints/scratch.pt')
train_utils.plot_history(history)

### Validation metrics on **seen** disasters
Confusion matrix + per-class precision/recall/F1. The held-out-disaster test is run
in notebook 04 where both models are compared head-to-head.

In [ ]:
train_utils.evaluate(model, val_dl, device, class_names=config.DAMAGE_CLASSES)